# Wiki recommender

## 0. Introduction

## 1. Setup

In [ ]:
import pandas as pd
import requests
import bs4
import re
import numpy as np
import time

In [5]:
SEED = 67

## 2. Data Collection (Crawling & Scraping)

In [7]:
class WikiScraper():
    
    def __init__(self, start_url_list, delay=3):
        self.headers = {'User-Agent': 'Recommender'}
        self.delay = delay # in seconds
        self.list_of_articles = []
        self.start_url_list = start_url_list
        
    def _fetch_article(self, url):
        try:
            response = requests.get(url, headers=self.headers, timeout=10)
        except requests.RequestException:
            return None
        if response.status_code != 200: return None

        parsed = bs4.BeautifulSoup(response.text, "html.parser")

        article = {"url": response.url}
        article["title"] = parsed.select_one("#firstHeading").get_text()
        article["content"] = " ".join(p.get_text() for p in parsed.select("#mw-content-text p"))
        links = self._extract_links(parsed)

        return article, links
        
    def _extract_links(self, parsed):
        links = parsed.find_all('a', attrs={'href': re.compile(r'^https://en\.wikipedia\.org/wiki/')})
        np.random.shuffle(links)
        return [link['href'] for link in links if ':' not in link['href'].split('/wiki/')[1]]
    
    def crawl(self, max_num_articles=50):
        np.random.seed(SEED)
        stack = list(reversed(self.start_url_list))
        visited = set()
        
        while stack and len(self.list_of_articles)<max_num_articles:
            url = stack.pop()
            if url in visited:
                continue
            
            visited.add(url)
            
            result = self._fetch_article(url)
            time.sleep(self.delay)
            
            if result is None:
                continue
            
            article, links = result
            stack.extend(links)
            
            if len(article["content"].split()) < 200:
                continue

            self.list_of_articles.append(article)
            
        print(f"Downloaded {len(self.list_of_articles)} articles, visited {len(visited)} URLs")
            
    def get_articles(self):
        return self.list_of_articles              

In [8]:
scraper = WikiScraper(
    ["https://en.wikipedia.org/wiki/Machine_learning",
     "https://en.wikipedia.org/wiki/Jazz"],
    delay=0.5,
)
scraper.crawl(max_num_articles=40)

df = pd.DataFrame(scraper.get_articles())
df["n_words"] = df["content"].str.split().str.len()

print(df[["title", "n_words"]])
print(df["content"].iloc[0][:500])

Downloaded 40 articles, visited 46 URLs
                                        title  n_words
0                            Machine learning     8827
1                                      MATLAB     1653
2                        Gaussian elimination     2966
3                         Category of modules      367
4                               Zero morphism      264
5                           Concrete category     1042
6                      Category (mathematics)     1752
7                                    Preorder     1359
8                          Transitive closure     1393
9                        Filter (mathematics)     2154
10                      Well-founded relation      930
11                                  Fuzzy set     1958
12                                Dialetheism     1175
13                              Gottlob Frege     2590
14                           Gemistos Plethon     2434
15                                       Mozi     4394
16                       

## 3. Stemming, lemmatization

In [9]:
import nltk

from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords


from nltk.tokenize import word_tokenize, wordpunct_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
_ =nltk.download('stopwords', quiet=True)

In [10]:
wordnet = WordNetLemmatizer()
porter = PorterStemmer()
stop = set(stopwords.words("english"))


In [11]:
def custom_stemmer(text):
    stems = []
    for x in word_tokenize(text.lower()):
        if x.isalpha() and x not in stop:
            stems.append(porter.stem(x))
    return stems

def custom_lemmatizer(text):
    lemms = []
    for x in word_tokenize(text.lower()):
        if x.isalpha() and x not in stop:
            lemms.append(wordnet.lemmatize(x))
    return lemms

In [12]:
df["tokens_stem"] = df["content"].apply(custom_stemmer)
df["tokens_lemma"] = df["content"].apply(custom_lemmatizer)

In [13]:
print(df["tokens_stem"].iloc[0][:20])
print(df["tokens_lemma"].iloc[0][:20])

['machin', 'learn', 'ml', 'field', 'studi', 'artifici', 'intellig', 'concern', 'develop', 'studi', 'statist', 'algorithm', 'learn', 'data', 'gener', 'unseen', 'data', 'thu', 'perform', 'task']
['machine', 'learning', 'ml', 'field', 'study', 'artificial', 'intelligence', 'concerned', 'development', 'study', 'statistical', 'algorithm', 'learn', 'data', 'generalize', 'unseen', 'data', 'thus', 'perform', 'task']


Vocabulary sizes

In [14]:
print(len(set(t for ts in df["tokens_stem"] for t in ts)),
      len(set(t for ts in df["tokens_lemma"] for t in ts))) 

5558 7588


In [17]:
df.to_parquet("articles.parquet", index=False)

## 4. Similarities

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [31]:
class Recommender():
    
    def __init__(self, df, text_column, preprocess, scraper=None):
        self.df = df
        self.text_column = text_column
        self.preprocess = preprocess
        if scraper is None:
            scraper = WikiScraper([])
        self.scraper = scraper
        self.vectorizer = TfidfVectorizer(min_df=2, max_df=0.8, ngram_range=(1,2), sublinear_tf=True)
        self.tfidf_matrix = None
        
    def fit(self):
        texts = self.df[self.text_column].apply(" ".join)
        self.tfidf_matrix = self.vectorizer.fit_transform(texts)
        self.title_to_idx = {title: i for i, title in enumerate(self.df["title"])}
        self.url_to_idx = {url: i for i, url in enumerate(self.df["url"])}
        
    def _new_vector(self, item):
        if item.startswith("http"):
            url = item
        else:
            url = "https://en.wikipedia.org/wiki/" + item.replace(" ", "_")

        result = self.scraper._fetch_article(url)
        if result is None:
            print(f"Not found: {item}")
            return None

        article, links = result
        tokens = self.preprocess(article["content"])
        return self.vectorizer.transform([" ".join(tokens)]).toarray()
        
    def recommend(self, items, top_n=10):
        indices = []
        new_vectors = []
        for item in items:
            if item in self.title_to_idx:
                indices.append(self.title_to_idx[item])
            elif item in self.url_to_idx:
                indices.append(self.url_to_idx[item])
            else:
                vector = self._new_vector(item)
                if vector is not None:
                    new_vectors.append(vector)
        if not indices and not new_vectors:
            return pd.DataFrame(columns=["title", "url", "score"])

        vectors = [self.tfidf_matrix[indices].toarray()] + new_vectors
        all_vectors = np.vstack(vectors)
        profile = all_vectors.mean(axis=0)
        scores = cosine_similarity([profile], self.tfidf_matrix)[0]
        scores[indices] = -1

        top_indices = np.argsort(scores)[::-1][:top_n]
        result = self.df.iloc[top_indices][["title", "url"]].copy()
        result["score"] = scores[top_indices]
        return result.reset_index(drop=True)
    

In [32]:
rec = Recommender(df, "tokens_lemma", preprocess=custom_lemmatizer)
rec.fit()
rec.recommend(["Jazz", "https://en.wikipedia.org/wiki/Saxophone"])

,title,url,score
0,B. F. Skinner,https://en.wikipedia.org/wiki/B._F._Skinner,0.425799
1,SAT,https://en.wikipedia.org/wiki/SAT,0.395893
2,Mozi,https://en.wikipedia.org/wiki/Mozi,0.351058
3,Machine learning,https://en.wikipedia.org/wiki/Machine_learning,0.330049
4,Amherst College,https://en.wikipedia.org/wiki/Amherst_College,0.322757
5,Gemistos Plethon,https://en.wikipedia.org/wiki/Gemistos_Plethon,0.291499
6,Gottlob Frege,https://en.wikipedia.org/wiki/Gottlob_Frege,0.278173
7,Common Object Request Broker Architecture,https://en.wikipedia.org/wiki/Common_Object_Re...,0.246313
8,Oracle Solaris,https://en.wikipedia.org/wiki/Solaris_(operati...,0.239019
9,Open Document Architecture,https://en.wikipedia.org/wiki/Open_Document_Ar...,0.215119
